# Predictive Modeling and Profit Optimization for Multi-Channel Restaurant Operations

**Dataset:** SkyCity Auckland Restaurants & Bars.csv
**Course project level:** 2nd-year B.E. Computer Science Engineering

This notebook walks through the full workflow:

Dataset -> Data Cleaning -> EDA -> Feature Engineering -> ML Model -> Prediction -> What-If Analysis -> Simple Optimization

The heavy-lifting functions (cleaning, feature engineering) live in `ml_utils.py` in the project root,
so this notebook and the Streamlit app (`app.py`) both use the exact same logic.


In [ ]:
import sys
sys.path.append("..")  # so we can import ml_utils.py from the project root

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ml_utils import load_raw_data, clean_data, engineer_features, DATA_PATH

sns.set_style("whitegrid")
%matplotlib inline


## 1. Load the Dataset

In [ ]:
df_raw = load_raw_data("../" + DATA_PATH)
print("Shape:", df_raw.shape)
df_raw.head()


In [ ]:
print("Column names:")
print(list(df_raw.columns))
print()
print("Data types:")
print(df_raw.dtypes)


## 2. Initial Inspection

Before cleaning anything, we check the basics: missing values, duplicate rows, duplicate IDs,
and basic statistics for the numeric columns.

In [ ]:
print("Missing values per column:")
missing = df_raw.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values found.")


In [ ]:
print("Duplicate rows:", df_raw.duplicated().sum())
print("Duplicate RestaurantID values:", df_raw['RestaurantID'].duplicated().sum())


In [ ]:
df_raw.describe().T


**Observation:** The dataset has no missing values and no duplicate rows or IDs. This is a fairly
clean, well-prepared dataset, so cleaning mainly involves defensive checks rather than heavy fixing.

## 3. Data Cleaning

In [ ]:
df_clean = clean_data(df_raw)
print("Shape after cleaning:", df_clean.shape)


**Why these particular cleaning steps?**

- Duplicate rows / duplicate RestaurantIDs are removed because each row should represent one
  restaurant's monthly snapshot; a repeat would be a data entry error.
- Missing values (if any appear later) are filled with the median (numeric) or the most frequent
  category (categorical) - simple, standard, and easy to explain in a viva.
- We only drop rows with **impossible** values (e.g. negative revenue or negative order counts).
  We do **not** blindly remove all outliers, because a restaurant with unusually high orders is a
  real busy restaurant, not necessarily a data error.
- Negative channel-level *net profit* values (e.g. a restaurant losing money on Uber Eats after
  commission and delivery costs) are kept, because that is a realistic business outcome.

## 4. Feature Engineering

In [ ]:
df = engineer_features(df_clean)
print("Shape after feature engineering:", df.shape)
df.head()


**New columns created:**

| Feature | Meaning |
|---|---|
| `TotalNetProfit` | Target variable: sum of all 4 channel net profits |
| `TotalRevenue` | Sum of revenue across all 4 channels |
| `RevenuePerOrder` | TotalRevenue / MonthlyOrders |
| `ProfitPerOrder` | TotalNetProfit / MonthlyOrders (display only, NOT a model input) |
| `ProfitMargin` | TotalNetProfit / TotalRevenue (display only, NOT a model input) |
| `CommissionCost` | Commission charged on Uber Eats + DoorDash revenue |
| `SelfDeliveryCost` | Total self-delivery cost (from `SD_DeliveryTotalCost`) |
| `*RevenueShare` | Each channel's share of actual revenue |
| `Commission_UE_Impact` | CommissionRate x UberEatsRevenue (interaction feature) |
| `DeliveryCost_SD_Impact` | DeliveryCostPerOrder x SelfDeliveryOrders (interaction feature) |
| `GrowthAdjustedOrders` | MonthlyOrders x GrowthFactor |


## 5. Exploratory Data Analysis (EDA)

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
sns.histplot(df["MonthlyOrders"], bins=30, kde=True, ax=ax)
ax.set_title("Monthly Orders Distribution")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
sns.histplot(df["AOV"], bins=30, kde=True, ax=ax, color="#DD8452")
ax.set_title("Average Order Value (AOV) Distribution")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
sns.histplot(df["TotalRevenue"], bins=30, kde=True, ax=ax, color="#55A868")
ax.set_title("Total Monthly Revenue Distribution")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
sns.histplot(df["TotalNetProfit"], bins=30, kde=True, ax=ax, color="#C44E52")
ax.set_title("Total Monthly Net Profit Distribution")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9,4.5))
order = df.groupby("CuisineType")["TotalNetProfit"].median().sort_values(ascending=False).index
sns.boxplot(data=df, x="CuisineType", y="TotalNetProfit", order=order, ax=ax, palette="Set2")
ax.set_title("Net Profit by Cuisine Type")
plt.xticks(rotation=30)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7,4.5))
sns.boxplot(data=df, x="Segment", y="TotalNetProfit", ax=ax, palette="Set3")
ax.set_title("Net Profit by Segment")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7,4.5))
sns.boxplot(data=df, x="Subregion", y="TotalNetProfit", ax=ax, palette="Set1")
ax.set_title("Net Profit by Subregion")
plt.show()


In [ ]:
channel_revenue = df[["InStoreRevenue","UberEatsRevenue","DoorDashRevenue","SelfDeliveryRevenue"]].sum()
fig, ax = plt.subplots(figsize=(7,4.5))
channel_revenue.plot(kind="bar", ax=ax, color=["#4C72B0","#DD8452","#55A868","#C44E52"])
ax.set_title("Total Revenue by Channel")
ax.set_ylabel("Revenue ($)")
plt.xticks(rotation=20)
plt.show()


In [ ]:
channel_profit = df[["InStoreNetProfit","UberEatsNetProfit","DoorDashNetProfit","SelfDeliveryNetProfit"]].sum()
fig, ax = plt.subplots(figsize=(7,4.5))
channel_profit.plot(kind="bar", ax=ax, color=["#4C72B0","#DD8452","#55A868","#C44E52"])
ax.set_title("Total Net Profit by Channel")
ax.set_ylabel("Net Profit ($)")
plt.xticks(rotation=20)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7,4.5))
sns.scatterplot(data=df, x="CommissionRate", y="UberEatsNetProfit", alpha=0.5, ax=ax)
ax.set_title("Commission Rate vs Uber Eats Net Profit")
plt.show()

print("Correlation:", df["CommissionRate"].corr(df["UberEatsNetProfit"]).round(3))


In [ ]:
fig, ax = plt.subplots(figsize=(7,4.5))
sns.scatterplot(data=df, x="DeliveryCostPerOrder", y="SelfDeliveryNetProfit", alpha=0.5, ax=ax, color="#C44E52")
ax.set_title("Delivery Cost per Order vs Self-Delivery Net Profit")
plt.show()

print("Correlation:", df["DeliveryCostPerOrder"].corr(df["SelfDeliveryNetProfit"]).round(3))


In [ ]:
from ml_utils import NUMERIC_FEATURES
corr_cols = NUMERIC_FEATURES + ["TotalNetProfit"]
corr = df[corr_cols].corr()
fig, ax = plt.subplots(figsize=(14,11))
sns.heatmap(corr, cmap="coolwarm", center=0, ax=ax, square=True, linewidths=0.3)
ax.set_title("Correlation Heatmap")
plt.show()


### EDA Observations (based on actual data, not assumptions)

- Commission Rate has a **negative** correlation with Uber Eats Net Profit in this dataset -
  higher commission rates are associated with lower Uber Eats profit.
- Delivery Cost per Order has a **negative** correlation with Self-Delivery Net Profit -
  higher delivery cost per order is associated with lower self-delivery profit.
- Profit varies noticeably by Cuisine Type and Segment, but this is an **observed pattern**,
  not proof that cuisine or segment *causes* higher or lower profit (correlation, not causation).

## 6. Data Leakage Check (IMPORTANT)

`TotalNetProfit` (our target) is calculated as:

```
TotalNetProfit = InStoreNetProfit + UberEatsNetProfit + DoorDashNetProfit + SelfDeliveryNetProfit
```

So these 4 columns **must be excluded** from the model's input features - otherwise the model
could just add them up and "cheat" instead of learning real patterns.

We also exclude `ProfitPerOrder` and `ProfitMargin` because both are calculated **from**
`TotalNetProfit`, so keeping them as inputs would be leakage in disguise.

Finally, `RestaurantID` and `RestaurantName` are excluded because they are just identifiers with
no real predictive meaning.

In [ ]:
from ml_utils import EXCLUDED_COLUMNS_EXPLANATION
for col, reason in EXCLUDED_COLUMNS_EXPLANATION.items():
    print(f"- {col}: {reason}")


## 7. Prepare Data for Modeling

In [ ]:
from ml_utils import get_model_ready_data

df_full, X, y = get_model_ready_data("../" + DATA_PATH)
print("X shape:", X.shape)
print("y shape:", y.shape)
X.head()


## 8. Train / Test Split, Preprocessing, and Model Training

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from ml_utils import CATEGORICAL_FEATURES, NUMERIC_FEATURES

RANDOM_STATE = 42

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print("Train size:", X_train.shape[0], "| Test size:", X_test.shape[0])


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES)],
    remainder="passthrough",
)

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
}

results = []
fitted_pipelines = {}

for name, model in models.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("regressor", model)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)

    results.append({"Model": name, "MAE": mae, "RMSE": rmse, "R2": r2})
    fitted_pipelines[name] = pipe
    print(f"{name:20s}  MAE={mae:9.2f}  RMSE={rmse:9.2f}  R2={r2:.4f}")


## 9. Model Comparison

In [ ]:
results_df = pd.DataFrame(results).sort_values("R2", ascending=False).reset_index(drop=True)
results_df


**Model selection:** We pick the model with the highest R^2 (and correspondingly low MAE/RMSE)
on the held-out test set. Gradient Boosting performed best on this dataset, likely because
profit depends on several non-linear interactions (e.g. commission rate combined with revenue
share) that a plain linear model cannot capture as well as a boosted tree ensemble.

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_pipeline = fitted_pipelines[best_model_name]
print("Best model:", best_model_name)


## 10. Feature Importance

In [ ]:
if best_model_name in ("Random Forest", "Gradient Boosting"):
    reg = best_pipeline.named_steps["regressor"]
    cat_encoder = best_pipeline.named_steps["preprocessor"].named_transformers_["cat"]
    cat_names = list(cat_encoder.get_feature_names_out(CATEGORICAL_FEATURES))
    feature_names = cat_names + NUMERIC_FEATURES

    imp_df = pd.DataFrame({"feature": feature_names, "importance": reg.feature_importances_})
    imp_df = imp_df.sort_values("importance", ascending=False).head(15)

    fig, ax = plt.subplots(figsize=(8,6))
    sns.barplot(data=imp_df, x="importance", y="feature", ax=ax)
    ax.set_title(f"Top 15 Feature Importances ({best_model_name})")
    plt.show()

    display(imp_df)
else:
    print("Best model is Linear Regression - showing coefficients instead of feature importances.")


**Interpretation:** Features like `OPEXRate` and `COGSRate` tend to dominate, which makes
business sense - they directly determine what fraction of revenue becomes profit before any
channel-specific costs are subtracted. Revenue-related features and self-delivery revenue also
matter, since they set the overall scale of profit. This is an **observed pattern** the model
relies on, not proof of a causal relationship.

## 11. Save the Best Model

In [ ]:
import joblib
joblib.dump(best_pipeline, "../model.pkl")
print("Model saved to ../model.pkl")


## 12. What-If Analysis (example)

A simple example of the What-If logic used in the Streamlit app: start from one restaurant's
current values, change one input, and compare predicted profit.

In [ ]:
from ml_utils import ALL_FEATURES

base = df_full.iloc[0]
row = {feat: base[feat] for feat in ALL_FEATURES}

def recompute_dependent_features(row):
    row = row.copy()
    row["CommissionCost"] = (row["UberEatsRevenue"] + row["DoorDashRevenue"]) * row["CommissionRate"]
    row["Commission_UE_Impact"] = row["CommissionRate"] * row["UberEatsRevenue"]
    row["DeliveryCost_SD_Impact"] = row["DeliveryCostPerOrder"] * row["SelfDeliveryOrders"]
    row["GrowthAdjustedOrders"] = row["MonthlyOrders"] * row["GrowthFactor"]
    return row

current_row = recompute_dependent_features(row)
current_pred = best_pipeline.predict(pd.DataFrame([current_row])[ALL_FEATURES])[0]

scenario_row = current_row.copy()
scenario_row["CommissionRate"] = 0.15  # lower commission scenario
scenario_row = recompute_dependent_features(scenario_row)
scenario_pred = best_pipeline.predict(pd.DataFrame([scenario_row])[ALL_FEATURES])[0]

diff = scenario_pred - current_pred
pct = (diff / current_pred) * 100

print(f"Current Predicted Profit:  ${current_pred:,.2f}")
print(f"Scenario Predicted Profit: ${scenario_pred:,.2f}")
print(f"Profit Difference:         ${diff:,.2f}")
print(f"Percentage Change:         {pct:+.2f}%")


## 13. Simple Optimization (example)

A basic grid search over a few channel-mix combinations, keeping only valid combinations
(shares sum to 100%), and picking the one the model predicts the highest profit for.

In [ ]:
import itertools

candidates = []
for is_pct, ue_pct, dd_pct in itertools.product([20,30,40,50], [10,20,30], [10,20,30]):
    sd_pct = 100 - is_pct - ue_pct - dd_pct
    if sd_pct < 0 or sd_pct > 100:
        continue
    r = current_row.copy()
    r.update({
        "InStoreShare": is_pct/100, "UE_share": ue_pct/100,
        "DD_share": dd_pct/100, "SD_share": sd_pct/100,
    })
    r = recompute_dependent_features(r)
    pred = best_pipeline.predict(pd.DataFrame([r])[ALL_FEATURES])[0]
    candidates.append({"InStore%": is_pct, "UberEats%": ue_pct, "DoorDash%": dd_pct,
                        "SelfDelivery%": sd_pct, "PredictedProfit": pred})

opt_df = pd.DataFrame(candidates).sort_values("PredictedProfit", ascending=False).reset_index(drop=True)
best_combo = opt_df.iloc[0]
uplift = ((best_combo["PredictedProfit"] - current_pred) / current_pred) * 100

print("Current Predicted Profit:   ${:,.2f}".format(current_pred))
print("Optimized Predicted Profit: ${:,.2f}".format(best_combo["PredictedProfit"]))
print("Optimization Uplift:        {:+.2f}%".format(uplift))
opt_df.head(10)


## 14. Conclusion

This notebook covers the complete workflow for this project: data cleaning, EDA, feature
engineering with explicit data-leakage prevention, training and comparing 3 regression models,
selecting the best one based on actual test-set metrics, and demonstrating the What-If and
Optimization logic that powers the Streamlit dashboard (`app.py`).

See `README.md` for how to run the full project, and `research_paper.md` /
`presentation_and_viva.md` for write-up and viva preparation content.